# TubeWatch Channel Tester
填写公开频道 URL 后从上到下运行。Tester 通过正式 CLI 保存发现记录；TubeScribe 处理默认关闭，必须显式启用。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("无法定位 TubeWatch 项目根目录。")

project_root = find_project_root(Path.cwd().resolve())
state_db = project_root / "data" / "tubewatch.sqlite3"
raw_directory = project_root / "output" / "raw"
cleaned_directory = project_root / "output" / "cleaned"
command_environment = dict(os.environ)
command_environment["PYTHONIOENCODING"] = "utf-8"

## 配置

In [ ]:
channel_url = "在这里填写 YouTube 频道链接"
limit = 10
run_tubescribe = False  # 只有改为 True 才会真实处理一条 pending 视频

In [ ]:
def run_cli(arguments):
    return subprocess.run(
        [sys.executable, "-m", "tubewatch", *arguments],
        cwd=project_root, capture_output=True, text=True, encoding="utf-8",
        env=command_environment,
    )

def failure_detail(result):
    return (result.stderr.strip() or result.stdout.strip()).replace("\r\n", "\n")

In [ ]:
if not channel_url.strip() or channel_url == "在这里填写 YouTube 频道链接":
    print("请先填写 channel_url，然后重新运行本单元格。")
else:
    arguments = ["check", channel_url, "--limit", str(limit), "--state-db", str(state_db), "--json"]
    first_run = run_cli(arguments)
    assert first_run.returncode == 0, failure_detail(first_run)
    first = json.loads(first_run.stdout)
    assert first["fetched_count"] <= limit
    assert 0 <= first["new_count"] <= first["fetched_count"]
    assert state_db.is_file() and state_db.stat().st_size > 0
    second_run = run_cli(arguments)
    assert second_run.returncode == 0, failure_detail(second_run)
    second = json.loads(second_run.stdout)
    assert second["new_count"] == 0 and second["new_videos"] == []
    print(f"本次读取：{first['fetched_count']}，新增：{first['new_count']}")
    print("频道检查与 SQLite 去重验证通过。")

## 可选：通过 TubeScribe 处理下一条 pending 视频

In [ ]:
if not run_tubescribe:
    print("TubeScribe 处理：SKIPPED（将 run_tubescribe 改为 True 后启用）")
else:
    result = run_cli([
        "process", "--limit", "1", "--state-db", str(state_db),
        "--raw-dir", str(raw_directory), "--cleaned-dir", str(cleaned_directory), "--json",
    ])
    assert result.stdout.strip(), failure_detail(result)
    payload = json.loads(result.stdout)
    assert payload["attempted_count"] in {0, 1}
    if payload["attempted_count"] == 0:
        assert result.returncode == 0
        print("没有待处理视频。")
    else:
        item = payload["results"][0]
        if item["status"] == "succeeded":
            assert result.returncode == 0
            assert Path(item["raw_path"]).is_file()
            assert Path(item["cleaned_path"]).is_file()
            assert Path(item["cleaned_path"]).read_text(encoding="utf-8").strip()
            print(f"处理成功：{item['title']}")
            print(f"清理文本：{item['cleaned_path']}")
        elif item["status"] == "no_subtitles":
            assert result.returncode == 0
            assert item["raw_path"] is None and item["cleaned_path"] is None
            assert item["error_message"]
            print(f"无可下载字幕，业务结果已正确记录：{item['title']}")
            print(item["error_message"])
        else:
            raise AssertionError(item.get("error_message") or payload)

## 无效输入验证（不访问网络）

In [ ]:
invalid_cases = [
    ([""], "channel_url 不能为空"),
    (["@bad/handle"], "有效的 YouTube 频道 handle"),
    (["https://example.com/not-a-channel"], "YouTube 频道 URL"),
    (["check", "https://example.com/not-a-channel"], "YouTube 频道 URL"),
    (["process", "--limit", "0"], "limit 必须是正整数"),
]
for arguments, expected_text in invalid_cases:
    result = run_cli(arguments)
    detail = failure_detail(result)
    assert result.returncode == 2, detail
    assert expected_text in detail and "Traceback" not in detail, detail
print("无效输入提示验证通过。")